### General settings

In [1]:

import geopandas as gpd
from shapely.geometry import Point
from collections import defaultdict
from itertools import product
from aocutils.special import UnionFind
from itertools import product
from copy import deepcopy
from functools import cache
import math
import requests
import networkx as nx
import matplotlib.pyplot as plt
from math import sqrt
from urllib.parse import urlparse, parse_qs
from itertools import chain, combinations
from math import sqrt, dist
from pathlib import Path
from operator import xor
import json

spanningscutoff = 100 # in case we want to use fewer station
use_tennet_stations = True
cutoff = 0.00001 # max distance between connections

# max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
max_distance = 0.006 # max distance between stations

accept_multiple_lines = False # for drawing
scale = 12 # the amount to scale with (higher --> more scaling down)
cluster_stations = True
spanning_kleuren = {
    380: 'red',
    150: 'blue',
    220: 'forestgreen',
    110: 'black',
    320: 'fuchsia',
    400: 'fuchsia'
}  

interconnectors = ["GNA-HGL380 W", "GNA-HGL380 Z", "DTC-NDR380 W", "DTC-NDR380 Z", "VYK-MBT380 W", "VYK-MBT380 Z", "RLL-ZVL380 G", "RLL-ZVL380 W", "MEE-DIL380 Z", "MEE-DIL380 W", "MBT-SDF380 Z", "MBT-OBZ380 W (SFK)", "EDC380-FDC300 Z"]

offshore = ["ZKL-BSA220 W", "ZKL-BSA220 O", "ZKL-BSB220 P", "HZL-HZB220 P", "HZL-HZA220 Z", "HZL-HZA220 W", "HNL-HNA220 Z", "HNL-HNA220 W", "HNL-HWA220 O", "HNL-HWA220 P", "WDC-EDR400 Nvt"]

In [2]:
def inspect_netschakel(netschaekl):
    for n in netschakels[netschaekl]:
        print(n, id2conn[n]['properties']['from_id'], id2conn[n]['properties']['to_id'])

In [3]:
# Laad de Nederland shapefile (EPSG:3035)
nl = gpd.read_file("netherlands_country_boundary/netherlands_Netherlands_Country_Boundary.shp")
country_shape = nl.unary_union  # merge all polygons once

def punt_in_nederland(lon, lat, boundary=country_shape):
    punt = Point(lon, lat)
    return boundary.contains(punt)

print(punt_in_nederland(4.895, 52.370))  # Amsterdam, True|
print(punt_in_nederland(6.14, 49.78))    # Buiten Nederland, False
print(punt_in_nederland(7.404059, 52.2884293))    # Buiten Nederland, False
print(punt_in_nederland(*[6.926667, 51.498889]))    # Buiten Nederland, False
print(punt_in_nederland(*[6.6308981, 51.0604542]))    # Buiten Nederland, False
print(punt_in_nederland(*[7.03359, 52.202448]))    # Buiten Nederland, False
print(punt_in_nederland(*[7.0325,52.2016]))    # Buiten Nederland, False

def distance_to_border(lon, lat, boundary=country_shape):
    punt = Point(lon, lat)
    if not boundary.contains(punt):
        return -punt.distance(boundary.boundary)
    else:
        return punt.distance(boundary.boundary)

def outside_border(lon, lat, boundary=country_shape, threshold=0.001, verbose=False):
    punt = Point(lon, lat)
    return not boundary.contains(punt)

# Compute centroids of Netherlands
x = float(nl.geometry.centroid.x.iloc[0])
y = float(nl.geometry.centroid.y.iloc[0])

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_24000\1331025071.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  country_shape = nl.unary_union  # merge all polygons once


True
False
False
False
False
False
False


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_24000\1331025071.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  x = float(nl.geometry.centroid.x.iloc[0])
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_24000\1331025071.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  y = float(nl.geometry.centroid.y.iloc[0])


### Get data from hoogspanningsnet.com

In [4]:
url = 'https://webkaart.hoogspanningsnet.com/layerdata.php?type=trms&zoom=8&bbox=4.086914062500001%2C51.23784668914442%2C17.869262695312504%2C53.68044193408406'

# URL parsen
parsed = urlparse(url)
query = parse_qs(parsed.query)
bbox = "3.315,50.775,7.226,53.576"

# Stations ophalen (stic)
def get_url(t, zoom=14):
    url = f"https://webkaart.hoogspanningsnet.com/layerdata.php?type={t}&zoom={zoom}&bbox={bbox}"
    return requests.get(url).json()

def getconn(num):
    if use_tennet_stations:
        return next(f for f in verbindingen['features'] if f['properties']['ID'] == str(num))
    else:
        return next(f for f in verbindingen['features'] if f['properties']['ID'] == 'v'+str(num))

def findstation(naam):
    return next(s for s in hoogspanning_stations['features'] if s['properties']['Naam']==naam)
hoogspanning_stations = get_url('stic')
hoogspanning_verbindingen = get_url('trvb')
hoogspanning_polygons = get_url('sttr', zoom=14) # werkt alleen bij zoom = 14 of gedetailleerder
knooppunten = get_url('trkp', zoom=14) 
masten = get_url('trms', zoom=14) 

for var in [hoogspanning_stations, hoogspanning_verbindingen, hoogspanning_polygons, knooppunten, masten]:
    print(len(var['features']))


2033
7598
2991
193
30534


### Remove stations and lines that are not relevant

In [5]:
hoogspanning_stations['features'] = [f for f in hoogspanning_stations['features'] if punt_in_nederland(*f['geometry']['coordinates']) and f['properties']['Spanning']>= spanningscutoff]
hoogspanning_polygons['features'] = [f for f in hoogspanning_polygons['features'] if punt_in_nederland(*f['geometry']['coordinates'][0][0]) and f['properties']['Spanning']>= spanningscutoff]
hoogspanning_verbindingen['features'] = [f for f in hoogspanning_verbindingen['features'] if (punt_in_nederland(*f['geometry']['coordinates'][0]) or punt_in_nederland(*f['geometry']['coordinates'][-1])) and f['properties']['Spanning']>= spanningscutoff]
print(len(hoogspanning_stations['features']), len(hoogspanning_polygons['features']), len(hoogspanning_verbindingen['features']))

458 593 1446


### Alternatively load TenneT Geojsons (I abandoned the Hoogspanningsnet data in favor of this)

In [6]:

with open("tennet/Hoogspanning_station.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)
with open("tennet/Opstijgpunt.geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
  
tennet = {}
tennet['features'] = []
for feature in chain(geo['features']): #, geo2['features']):
    props = feature["properties"]
    if "SE_FLD55_SPANNINGSNIVEAU" in props:
        props["Spanning"] = props.pop("SE_FLD55_SPANNINGSNIVEAU")
    elif "SE_FLD18_SPANNINGSNIVEAU" in props:
        props["Spanning"] = props.pop("SE_FLD18_SPANNINGSNIVEAU")
        
    if "SE_FLD24_OBJECTOMSCHRIJVING" in props:
        props["Naam"] = props.pop("SE_FLD24_OBJECTOMSCHRIJVING")
        props["Osp"] = False
    elif 'SE_FLD13_OBJECTID' in props: # opstijgpunt
        props["Naam"] = props.pop("SE_FLD13_OBJECTID")
        props["Osp"] = True
        
        
    if feature["geometry"] is not None and props['Naam'] is not None:
        idx = props["Naam"].rfind(' ')
        # print(props["Naam"])
        if idx > 8 and props["Naam"].startswith('Station'): props["Naam"] = props["Naam"][8:idx]
        if props["Naam"][-1].isdigit():
            idx = props["Naam"].rfind(' ')
            props["Naam"] = props["Naam"][:idx]
        if props["Spanning"] > spanningscutoff or props['Naam'] == 'Oudehaske':
            tennet['features'].append(feature)
            if 'coordinates' not in feature['geometry']:
                print(feature)
with open("tennet/Hoogspanning_kabel_(ondergronds).geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)
with open("tennet/Hoogspanning_leiding_(bovengronds).geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
tennet_verbindingen = {
    "type": "FeatureCollection",
    "features": geo["features"] + geo2["features"]
}
for feature in tennet_verbindingen['features']:
    feature['properties']['ID'] = feature['properties']['SE_FLD32_OBJECTID'] if 'SE_FLD32_OBJECTID' in feature['properties'] else feature['properties']['SE_FLD33_OBJECTID']
    feature['properties']['Spanning'] = feature['properties']['SE_FLD38_SPANNINGSNIVEAU'] if 'SE_FLD38_SPANNINGSNIVEAU' in feature['properties'] else feature['properties']['SE_FLD39_SPANNINGSNIVEAU']
    feature['properties']['Netschakel'] = feature['properties']['SE_FLD27_NETSCHAKELID'] if 'SE_FLD27_NETSCHAKELID' in feature['properties'] else feature['properties']['SE_FLD28_NETSCHAKELID']
    
tennet_verbindingen['features'] = [f for f in tennet_verbindingen['features'] if f['geometry']['type'] == 'LineString' and feature['properties']['Spanning'] > spanningscutoff]

In [7]:
with open("tennet/Opstijgpunt.geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
osp = [o['geometry']['coordinates'][0][0] for o in geo2['features']]
len(osp) # amount of opspanportalen

161

### Determine stationsnamen of polygons

In [8]:
if use_tennet_stations:
    stations = deepcopy(tennet)
    verbindingen = deepcopy(tennet_verbindingen)
    verbindingen['features'] = [v for v in verbindingen['features'] if v['properties']['Spanning'] > spanningscutoff]
    station_polygons = deepcopy(tennet)
else:
    stations = deepcopy(hoogspanning_stations)
    verbindingen = deepcopy(hoogspanning_verbindingen)
    station_polygons = hoogspanning_polygons
    
id2conn = {f['properties']['ID']: f for f in verbindingen['features']}
not_found = {f['properties']['Naam'] for f in stations['features']}

if not use_tennet_stations:
    cnt = 1
    matched = 0
    for poly in station_polygons['features']:
        coords = poly['geometry']['coordinates'][0]
        avg_x = sum(p[0] for p in coords)/len(coords)
        avg_y = sum(p[1] for p in coords)/len(coords)
        poly_point = (avg_x, avg_y)
        
        # Vind het dichtstbijzijnde station
        min_dist = float('inf')
        closest_station_name = None
        for station in stations['features']:
            if use_tennet_stations:
                station_point = station['geometry']['coordinates'][0][0]
            else:
                station_point = station['geometry']['coordinates']
                
            d = dist(poly_point, station_point)
            if d < min_dist:
                min_dist = d
                closest_station = station
        
        if min_dist <= max_distance_line_station:
            poly['properties']['Naam'] = closest_station['properties']['Naam']
            not_found.discard(closest_station['properties']['Naam'])
            matched +=1 
        else:
            cnt +=1
    print(matched)
    print(len(not_found))
    not_found # Nuon Magnumcentrale is wel even interessant om in de gaten te houden



In [9]:
# 320 and 400
spanning = defaultdict(int)
for f in verbindingen['features']:
    spanning[f['properties']['Spanning']] += 1
spanning
    

defaultdict(int,
            {150: 42874, 110: 21805, 380: 21191, 220: 5939, 320: 33, 400: 7})

In [10]:
id2conn = {f['properties']['ID']: f for f in verbindingen['features']}
netschakels = defaultdict(list)
if use_tennet_stations:
    for v in verbindingen['features']:
        netschakels[v['properties']['Netschakel']].append(v['properties']['ID'])
        
coortonet = defaultdict(list)
for n in netschakels:
    for conn in netschakels[n]:
        c = id2conn[conn]['geometry']['coordinates'][0]
        coortonet[tuple(c)].append((n,conn))
        c = id2conn[conn]['geometry']['coordinates'][-1] # only the extremeties
        coortonet[tuple(c)].append((n,conn))
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>1}
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>2}
print(len(coortonet))

95238
88112
271


### Determine start and endlocation of connections

In [11]:


def point_in_polygon_strict(point, polygon):
    """
    Bepaal of een punt binnen een polygon ligt met ray-casting.
    point: (x, y)
    polygon: lijst van (x, y) tuples
    """
    x, y = point
    inside = False
    n = len(polygon)
    
    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y)*(p2x - p1x)/(p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

def point_line_distance(px, py, x1, y1, x2, y2):
    """Afstand van punt (px,py) tot lijnstuk (x1,y1)-(x2,y2)."""
    dx, dy = x2 - x1, y2 - y1
    if dx == dy == 0:
        return math.hypot(px - x1, py - y1)  # lijnstuk is een punt
    t = max(0, min(1, ((px - x1) * dx + (py - y1) * dy) / (dx*dx + dy*dy)))
    nx, ny = x1 + t*dx, y1 + t*dy
    return math.hypot(px - nx, py - ny)


def point_in_polygon_less_strict(point, polygon, epsilon=0.001):
    """
    Bepaal of een punt binnen een polygon ligt met ray-casting,
    of binnen 'epsilon' afstand van de rand.
    """
    x, y = point
    inside = False
    n = len(polygon)
    
    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        # check afstand tot rand
        if point_line_distance(x, y, p1x, p1y, p2x, p2y) <= epsilon:
            return True
        p1x, p1y = p2x, p2y

# Eerst bounding boxes van alle polygons berekenen
# Precompute bounding boxes and polygon lengths once
polygon_bboxes = []
for feature in station_polygons['features']:
    if 'Naam' in feature['properties']:
        coords = feature['geometry']['coordinates'][0]
        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
        polygon_bboxes.append({
            'ID': feature['properties']['Naam'],
            'min_x': min(xs),
            'max_x': max(xs),
            'min_y': min(ys),
            'max_y': max(ys),
            'polygon': coords,
            'spanning': feature['properties']['Spanning'],
            'n': len(coords)  # precompute once
        })

    
@cache
def find_polygon(x, y, epsilon=0.001):
    
    """Return the polygon ID containing the point, strict first, then epsilon margin"""
    # if use_tennet_stations: epsilon=0.000
    for bbox in polygon_bboxes:
        # Quick bounding box check
        if not (bbox['min_x'] <= x <= bbox['max_x'] and bbox['min_y'] <= y <= bbox['max_y']):
            continue  # skip polygons that can't contain the point
        # Strict point-in-polygon
        inside = False
        n = bbox['n']
        p1x, p1y = bbox['polygon'][0]
        for i in range(n + 1):
            p2x, p2y = bbox['polygon'][i % n]
            if y > min(p1y, p2y):
                if y <= max(p1y, p2y):
                    if x <= max(p1x, p2x):
                        if p1y != p2y:
                            xinters = (y - p1y)*(p2x - p1x)/(p2y - p1y) + p1x
                        if p1x == p2x or x <= xinters:
                            inside = not inside
            # less strict: check epsilon distance to edge
            dx, dy = p2x - p1x, p2y - p1y
            if dx == dy == 0:
                d = math.hypot(x - p1x, y - p1y)
            else:
                t = max(0, min(1, ((x - p1x) * dx + (y - p1y) * dy) / (dx*dx + dy*dy)))
                nx, ny = p1x + t*dx, p1y + t*dy
                d = math.hypot(x - nx, y - ny)
            if d <= epsilon:
                # print(d)
                return bbox['ID']
            p1x, p1y = p2x, p2y

        if inside:
            return bbox['ID']

    return False

# these lines should not be taken into account, since the lines don't end at this station, but pass overhead
connection_exclude = {'848719', '848727', '848731', '848734', '848725', '848717'} # Hengelo Marssteden has an overhead passing 380kV line


# Controleer elk punt van elke verbinding
for feature in verbindingen['features']:
    conn_list = []
    verbinding_id = feature['properties']['ID']
    for point in feature['geometry']['coordinates']:
        if (res:=find_polygon(*point)):
            conn_list.append(res)
        
    if len(conn_list) > 1 and verbinding_id not in connection_exclude:
        color = spanning_kleuren.get(feature['properties']['Spanning'], 'grey')
    if verbinding_id not in connection_exclude:
        feature['properties']['from_id'] = find_polygon(*feature['geometry']['coordinates'][0])
        feature['properties']['to_id'] = find_polygon(*feature['geometry']['coordinates'][-1])
        feature['properties']['connections'] = set(conn_list)
    else:
        feature['properties']['from_id'] = False
        feature['properties']['to_id'] = False

### at this point, not all lines and cables have a start and endstation, lets fix that

In [12]:
def parallellines(vid1, vid2):
    start1 = vid1['geometry']['coordinates'][0]
    end1 = vid1['geometry']['coordinates'][-1]
    start2 = vid2['geometry']['coordinates'][0]
    end2 = vid2['geometry']['coordinates'][-1]
    options = product([start1, end1], [start2, end2])
    return sum(dist(*o) < (cutoff * 1) for o in options) >= 2

In [26]:
def process_connections(results, netschakel):
    if netschakel == 'MLBK-BSDL150 Z':
        print('hih', results)
    idcounter = 1
    found = []
    unique_connections = []
    for option in results:
        start = id2conn[option[0]]['properties']['from_id'] if id2conn[option[0]]['properties']['from_id'] else id2conn[option[0]]['properties']['to_id'] 
        end = id2conn[option[-1]]['properties']['from_id'] if id2conn[option[-1]]['properties']['from_id'] else id2conn[option[-1]]['properties']['to_id']
        if start != end and sorted([start, end]) not in found:
            found.append(sorted([start, end]))
            unique_connections.append(deepcopy(id2conn[option[0]]))
            
            if not id2conn[option[0]]['properties']['from_id']: 
                unique_connections[-1]['geometry']['coordinates'] = unique_connections[-1]['geometry']['coordinates'][::-1]
            unique_connections[-1]['properties']['ID'] += '_' + str(idcounter)
            idcounter += 1
            unique_connections[-1]['properties']['from_id'] = start
            unique_connections[-1]['properties']['to_id'] = end
            for lineid in option[1:]:
                if id2conn[lineid]['geometry']['coordinates'][0] == unique_connections[-1]['geometry']['coordinates'][-1]:
                    unique_connections[-1]['geometry']['coordinates'] += id2conn[lineid]['geometry']['coordinates'][1:]
                else:
                    unique_connections[-1]['geometry']['coordinates'] += id2conn[lineid]['geometry']['coordinates'][::-1][1:]
    return unique_connections
            

In [27]:

def dfs(cur, path, goals, splits):
    results = []
    if id2conn[cur]['geometry']['coordinates'][0] in splits or id2conn[cur]['geometry']['coordinates'][-1] in splits:
        return [path]
    for neighbor in neigh[cur]:
        if neighbor not in path:
            if neighbor in goals:
                return [path + [neighbor]]
            else:
                path.append(neighbor)
                for r in dfs(neighbor, path, goals, splits):
                    results.append(r)
                assert path.pop() == neighbor
    return results

# certain splits we want to ignore       
forbidden_split = [[5.469404028332668,52.33157631469825],
                   [4.67473246782226, 52.4597878584342]
                 ]
forbidden_split = osp

def too_close_to_forbidden_splits(c):
    # return False
    # print(c)
    return min([dist(f, c) for f in forbidden_split]) < max_distance
# too_close_to_forbidden_splits([4.67473246782226, 52.4597878584342])
    
# make a dict
def make_coor2id(netschakel):
    coor2id = defaultdict(list)
    for lineid in netschakels[netschakel]:
        if id2conn[lineid]['properties']['from_id'] and id2conn[lineid]['properties']['to_id']:
            # print('line ignored since within a station', id2conn[lineid]['properties']['from_id'], id2conn[lineid]['properties']['to_id'])
            continue
        
        
        coor2id[tuple(id2conn[lineid]['geometry']['coordinates'][0])].append(lineid)
        coor2id[tuple(id2conn[lineid]['geometry']['coordinates'][-1])].append(lineid)
    return coor2id

def make_neighbors(coor2id):
    conn = defaultdict(set)
    for connected in coor2id.values():
        for c1 in connected:
            for c2 in connected:
            
                if c1 != c2:
                    conn[c1].add(c2)
    return conn

In [41]:

presentstations = {s['properties']['Naam'] for s in stations['features']}
seen = {}
splitcounter = 0
interconnectcounter = 9999
finalset = {}
finalset['features'] = []

for netschakel in netschakels:
    splitcounter += 1
    results = []
    coor2id = make_coor2id(netschakel) # per coordinate al the lines
    neigh = make_neighbors(coor2id) # per line all it's neighbors (on both sides)
    splits = [k for k,v in coor2id.items() if len(v) > 2 and not too_close_to_forbidden_splits(k)]
    
    if netschakel in (interconnectors + offshore) and str(interconnectcounter) not in presentstations:
        distances = []
        for line in netschakels[netschakel]:
            coor = id2conn[line]['geometry']['coordinates'][0]
            distances.append((distance_to_border(*coor), coor, line, 'from_id'))
            coor = id2conn[line]['geometry']['coordinates'][-1]
            distances.append((distance_to_border(*coor), coor, line, 'to_id'))
        distances.sort()
        furthest_dist, furthest_coor, furthest_line, fromto = distances[0]
        
        presentstations.add(str(interconnectcounter))        
        stations['features'].append(deepcopy(stations['features'][0]))
        stations['features'][-1]['geometry']['coordinates'] = [[furthest_coor]]
        stations['features'][-1]['properties']['Naam'] = str(interconnectcounter)
        stations['features'][-1]['properties']['Spanning'] = 380
        
        id2conn[furthest_line]['properties'][fromto] = str(interconnectcounter)
        interconnectcounter -= 1
        
        
        
    # get starting lines
    for lineid in netschakels[netschakel]:
        line = id2conn[lineid]
        start = line['properties']['from_id']
        end = line['properties']['to_id']  
        # determine if line are starting points within a station (not fully within a station)  
        if line['geometry']['coordinates'][0] in splits:
            line['properties']['from_id'] = str(splitcounter)
        if line['geometry']['coordinates'][-1] in splits:
            line['properties']['to_id'] = str(splitcounter)
        if splits and str(splitcounter) not in presentstations:   
            presentstations.add(str(splitcounter))        
            stations['features'].append(deepcopy(stations['features'][0]))
            stations['features'][-1]['geometry']['coordinates'] = [[splits[0]]]
            stations['features'][-1]['properties']['Naam'] = str(splitcounter)
            stations['features'][-1]['properties']['Spanning'] = 380
        
    starting = []
    for lineid in netschakels[netschakel]:
        line = id2conn[lineid]
        start = line['properties']['from_id']
        end = line['properties']['to_id']  
        if xor(start is False, end is False): 
            starting.append(lineid)

    if netschakel == 'MLBK-BSDL150 Z':
        print('jojo', starting, splits, neigh)
        for k,v in neigh.items():
            if len(v) != 2:
                v = v.pop()
                print(k,v, id2conn[k]['geometry']['coordinates'])
    for startinglineid in starting:
        results += dfs(startinglineid, [startinglineid], starting, splits)
        
    finalset['features'] += process_connections(results, netschakel)
         
print(len(finalset['features']), len(stations['features']))

jojo ['700451', '700468', '700471', '708327', '708331', '708309'] [] defaultdict(<class 'set'>, {'700566': {'700582', '700554'}, '700554': {'708951', '700566'}, '700582': {'700566', '700594'}, '700568': {'700580', '700556'}, '700556': {'708955', '700568'}, '700580': {'700592', '700568'}, '700570': {'700578', '700558'}, '700558': {'708959', '700570'}, '700578': {'700590', '700570'}, '703038': {'703017', '703065'}, '703017': {'703038', '702747'}, '703065': {'703089', '703038'}, '707743': {'707564', '707769'}, '707564': {'707515', '707743'}, '707769': {'707743', '707803'}, '707282': {'705112', '707292'}, '705112': {'707282', '705079'}, '707292': {'707311', '707282'}, '707284': {'707290', '705108'}, '705108': {'705081', '707284'}, '707290': {'707284', '707313'}, '707286': {'707288'}, '707288': {'707286', '707309'}, '704412': {'704433', '704393'}, '704393': {'704412'}, '704433': {'704448', '704412'}, '704778': {'704799', '704772'}, '704772': {'704778', '704746'}, '704799': {'704778', '70480

In [ ]:
704393 704412 [[5.94743431901047, 51.0263602057971], [5.94812803614873, 51.0235156498035]]
706849 706838 [[5.94743431623633, 51.0263602121028]]

In [ ]:
707286 707288 [[5.99486283381978, 50.9599197760947], [5.9979947499009, 50.9587917698107]]
705110 705077 [5.99486284227642, 50.959919769758]]

In [40]:
id2conn['707286']

{'type': 'Feature',
 'id': 78845,
 'geometry': {'type': 'LineString',
  'coordinates': [[5.99486283381978, 50.9599197760947],
   [5.9979947499009, 50.9587917698107]]},
 'properties': {'SE_FLD0_MHSKEY': '707286|50053',
  'SE_FLD3_BEDRIJFSSTATUS': 'In bedrijf',
  'SE_FLD6_BOUWJAAR': -189388800000,
  'ESRI_OID': 78845,
  'SE_FLD28_NETSCHAKELID': 'MLBK-BSDL150 Z',
  'SE_FLD33_OBJECTID': '707286',
  'SE_FLD35_OBJECTTYPE': 'HSleidingdeel',
  'SE_FLD39_SPANNINGSNIVEAU': 150,
  'Shape__Length': 253.31858923066727,
  'ID': '707286',
  'Spanning': 150,
  'Netschakel': 'MLBK-BSDL150 Z',
  'from_id': False,
  'to_id': False,
  'connections': set()}}

In [30]:
inspect_netschakel('MLBK-BSDL150 Z')

700566 False False
700568 False False
700570 False False
703038 False False
707743 False False
707282 False False
707284 False False
707286 False False
704412 False False
704393 False False
704778 False False
926455 False False
707763 False False
707385 False False
707390 False False
702292 False False
703052 False False
705534 False False
705535 False False
705541 False False
704671 False False
704672 False False
702721 False False
702258 False False
702260 False False
703865 False False
707309 False False
702295 False False
702297 False False
704116 False False
704118 False False
708173 False False
708175 False False
704797 False False
704799 False False
704998 False False
926456 False False
701634 False False
706849 False False
703517 False False
703519 False False
705043 False False
705045 False False
707916 False False
708161 False False
708163 False False
704714 False False
704920 False False
704922 False False
707739 False False
704306 False False
705417 False False
707769 False

### Count the stations how many connections they have

In [16]:
verbindingen = finalset

if cluster_stations:
    from collections import defaultdict
    conn = defaultdict(int)
    for v in verbindingen['features']:
        conn[v['properties']['from_id']] += 1
        conn[v['properties']['to_id']] += 1
    conn

In [17]:
priority = ['Eemshaven', 'Borssele', 'Diemen', 'Geervliet'] # these stations will be more important than others in their vicinity

station2spanning = {s['properties']['Naam']: s['properties']['Spanning'] for s in stations['features']}
if cluster_stations:
    from collections import defaultdict
    conn = defaultdict(int)
    for v in verbindingen['features']:
        conn[v['properties']['from_id']] += 1
        conn[v['properties']['to_id']] += 1
    conn


    ufstation = UnionFind([])
    for idx, s1 in enumerate(stations['features']):
        for idx2, s2 in enumerate(stations['features'][idx+1:], start=idx+1):
            if use_tennet_stations:
                if dist(s1['geometry']['coordinates'][0][0], s2['geometry']['coordinates'][0][0]) < max_distance:
                    ufstation.union(s1['properties']['Naam'],s2['properties']['Naam'])
            else:
                if dist(s1['geometry']['coordinates'], s2['geometry']['coordinates']) < max_distance:
                    ufstation.union(s1['properties']['Naam'],s2['properties']['Naam'])
    station2mainstation = {}
    removed = set()

    for g in ufstation.groups():
        sortedgroup = sorted(g, key=lambda x: (x not in priority, x[0].isnumeric(), -station2spanning[x], -conn[x]))
        print(sortedgroup)
        for s in sortedgroup[1:]:
            station2mainstation[s] = sortedgroup[0]
            removed.add(s)
        
else:
    removed = set()
    station2mainstation = {}

['Borssele', 'Zeeuwse Kust Landstation']
['Hengelo Weideweg']
['Vierverlaten']
['Lelystad']
['Diemen', 'Diemer Vijfhoek']
['Ens']
['Krimpen a/d IJssel']
['Eemshaven', 'Eemshaven Oudeschip', 'Eemshaven Converterstation 380 kV', 'Eemshaven Synergieweg 380kV', 'Waddenweg Converterstation 380 kV', 'Eemshaven Comp. en Filteren', 'Robbenplaat', 'Oostpolder', 'Eemshaven Oost']
['Groningen Hunze', 'Groningen Bornholmstraat']
['Bergum']
['Louwsmeer']
['Doetinchem', 'Langerak']
['Wijk aan Zee', 'Hollandse Kust Noord Landstation']
['Rilland', '157']
['Zeyerveen']
['Meeden']
['Westerlee', 'De Lier']
['Zwolle', 'Hessenweg', 'Zwolle Hessenweg']
['Borculo', 'Borculo Berkel']
['Beersdal', 'Huskensweg']
['Boekend', '825']
['Geervliet', 'Geervliet Noorddijk']
['Boxmeer', '871']
['Breukelen Kortrijk']
['Hengelo', 'Hengelo Oele']
['Amsterdam Noord Klaprozenweg', 'Amsterdam Noord Papaverweg']
['Wateringen']
['Anna Paulowna', 'BBL Gasunie']
['Vijfhuizen']
['Maasvlakte']
['Eindhoven', 'Eindhoven Oost']
['Bij

### Make data for loom

In [24]:
# Helper to see new connections to and from a specific station

f = 'Beersdal'
for edge in verbindingen['features']:
    if edge['properties']['from_id'] == f or edge['properties']['to_id']  == f:
        print(edge['properties']['Spanning'], edge['properties']['from_id'], edge['properties']['to_id'], edge['properties']['ID'])

150 Treebeek Beersdal 793683_1
150 Terwinselen Beersdal 776914_1
150 Terwinselen Beersdal 777139_1
150 Beersdal Huskensweg 986314_1


In [23]:
inspect_netschakel('MLBK-BSDL150 Z')

700566 False False
700568 False False
700570 False False
703038 False False
707743 False False
707282 False False
707284 False False
707286 False False
704412 False False
704393 False False
704778 False False
926455 False False
707763 False False
707385 False False
707390 False False
702292 False False
703052 False False
705534 False False
705535 False False
705541 False False
704671 False False
704672 False False
702721 False False
702258 False False
702260 False False
703865 False False
707309 False False
702295 False False
702297 False False
704116 False False
704118 False False
708173 False False
708175 False False
704797 False False
704799 False False
704998 False False
926456 False False
701634 False False
706849 False False
703517 False False
703519 False False
705043 False False
705045 False False
707916 False False
708161 False False
708163 False False
704714 False False
704920 False False
704922 False False
707739 False False
704306 False False
705417 False False
707769 False

In [19]:
# these connections are ignored because I don't want them to be drawn
forbiddenlines = "2387372_1 2387343_2 548828_1 495468_1 495462_3 263392_2 1706983_1 1940482_3 1706977_2 742566_1 1709656_1 1744529_1 1621460_3 1621455_3".split()

In [20]:
verbindingen['features'] = sorted(verbindingen['features'], key = lambda x: -x['properties']['Spanning'])
forbidden = ['Enecogen']
forbidden = []
def normalize(coor, scale = scale):
    dx = coor[0] - x
    dy = coor[1] - y
    
    return [x + dx/scale, y + dy/scale]
   
from pyproj import Transformer
# Example: WGS84 (lon/lat) → UTM zone 31N (meters)
proj_to_meters = Transformer.from_crs("EPSG:4326", "EPSG:32631", always_xy=True)
proj_to_lonlat = Transformer.from_crs("EPSG:32631", "EPSG:4326", always_xy=True)

# Original normalize function in lon/lat
def normalize(coor, scale=scale):
    # 1. Convert lon/lat to meters
    mx, my = proj_to_meters.transform(coor[0], coor[1])
    mx0, my0 = proj_to_meters.transform(x, y)
    
    # 2. Do your scaling in meters
    dx = mx - mx0
    dy = my - my0
    mx_new = mx0 + dx / scale
    my_new = my0 + dy / scale
    
    # 3. Convert back to lon/lat
    lon_new, lat_new = proj_to_lonlat.transform(mx_new, my_new)
    return [lon_new, lat_new]
 
    
accepted = {f['properties']['Naam'] for f in stations['features'] if f['properties']['Naam'] not in forbidden} 

loom_nodes = {
    "type": "FeatureCollection",
    "features": []
}

for node in stations['features']:
    if node['properties']['Naam'] in accepted:
        loom_node = {
            "type": "Feature",
            "geometry": node['geometry'].copy(),
            "properties": {
                "id": node['properties']['Naam'],
                "station_label": ' ' + node['properties']['Naam'] + '  '
            }
        }
        # for d in ['deg', 'deg_in', 'deg_out']:
        #     loom_node['properties'][d] =G.degree[node['properties']['Naam']]
        if use_tennet_stations:
            loom_node["geometry"]["coordinates"] = normalize(loom_node["geometry"]["coordinates"][0][0])
            loom_node['geometry']['type'] = 'Point'
        else:
            loom_node["geometry"]["coordinates"] = normalize(loom_node["geometry"]["coordinates"])
            
        loom_nodes['features'].append(loom_node)

# Save to JSON
with open("loom_nodes.json", "w") as f:
    json.dump(loom_nodes, f, indent=2)

print("Conversion done! Saved as loom_nodes.json")

# Convert edges to Loom format
loom_edges = {
    "type": "FeatureCollection",
    "features": []
}

def sample(lst, size=10):
    newlist = lst[::size]
    if newlist[-1] != lst[-1]:
        newlist.append(lst[-1])
    return newlist

seen = set()
for edge in verbindingen['features']:
    if edge['properties']['ID'] == '2387372':
        print(edge['properties'])
    if 'from_id' in edge['properties'] and edge['properties']['to_id'] and edge['properties']['Spanning'] > spanningscutoff and edge['properties']['ID'] not in forbiddenlines:
        
        spanning = str(round(edge['properties']['Spanning']))
        loom_edge = {
            "type": "Feature",
            "geometry": edge['geometry'].copy(),
            "properties": {
                "identifier": edge['properties']['ID'],
                "from": station2mainstation.get(edge['properties']['from_id'], edge['properties']['from_id']),
                "to": station2mainstation.get(edge['properties']['to_id'], edge['properties']['to_id']),
                "dbg_lines": str(round(edge['properties']['Spanning'])),
                "spanning": spanning,
                "lines": [{
                "color": spanning_kleuren.get(int(spanning), 'grey'),
                "id": spanning,
                }]
            }
        }
        
        loom_edge["geometry"]["coordinates"] = [normalize(c) for c in loom_edge["geometry"]["coordinates"]]
        # loom_edge["geometry"]["coordinates"] = loom_edge["geometry"]["coordinates"][:25] + loom_edge["geometry"]["coordinates"][-25:]
        loom_edge["geometry"]["coordinates"] = [loom_edge["geometry"]["coordinates"][0], loom_edge["geometry"]["coordinates"][-1]]
        
        if True:
            # print(loom_edge['properties'])
            if loom_edge['properties']['from'] != loom_edge['properties']['to']: # no self loops
                loom_edge["properties"]["fromto"] = tuple(sorted((loom_edge['properties']['from'], loom_edge['properties']['to'], loom_edge['properties']['spanning'])))
                if loom_edge["properties"]["fromto"] not in seen:
                    seen.add(loom_edge["properties"]["fromto"])
                    loom_edges['features'].append(loom_edge)
                else:
                    if accept_multiple_lines:
                        for e in loom_edges['features']:
                            if e['properties']['fromto'] == loom_edge['properties']['fromto']:
                                e['properties']['lines'].append(loom_edge['properties']['lines'][0])
                                break
        else:
            pass
    else:
        pass

# Save to JSON
with open("loom_edges.json", "w") as f:
    json.dump(loom_edges, f, indent=2)

print("Edges converted to Loom format!")

loom_all = {
    "type": "FeatureCollection",
    "features": loom_nodes['features'] + loom_edges['features']
}

# Optionally save to a file
with open("loom_combined.json", "w") as f:
    json.dump(loom_all, f, indent=2)

wsl_path = Path(r"\\wsl.localhost\Ubuntu-24.04\home\jesse\loom\examples\netkaart3.json")

# Write JSON with LF line endings
with wsl_path.open("w", encoding="utf-8", newline="\n") as f:
    json.dump(loom_all, f, indent=2, ensure_ascii=False)
    f.write("\n")  # make sure file ends with a newline
print("Nodes and edges combined into one Loom JSON!")

# cat examples/netkaart3.json | docker run -i loom loom | docker run -i loom octi | docker run -i loom transitmap -l > netkaart-octilinear.svg

Conversion done! Saved as loom_nodes.json
Edges converted to Loom format!
Nodes and edges combined into one Loom JSON!


### Deploy

In [21]:
# Optimal settings for big map

# import geopandas as gpd
# from shapely.geometry import Point
# import json
# from collections import defaultdict
# spanningscutoff = 100
# use_tennet_stations = True
# cutoff = 0.00001 # max distance between connections
# # max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
# max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
# max_distance = 0.006 # max distance between stations

# accept_multiple_lines = False # for drawing
# scale = 11 # the amount to scale with (higher --> more scaling down)
# cluster_stations = True
# spanning_kleuren = {
#     380: 'red',
#     150: 'blue',
#     220: 'forestgreen',
#     110: 'black'
# }  

# cat examples/netkaart3.json | ./build/topo | ./build/loom | ./build/octi -g 400 --geo-pen 0.1 --nd-move-pen 0.5 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > finaloutput.svg

In [22]:
# in WSL first open a docker container with the London Tube font
docker run -it \
  -v "$PWD":/workspace \
  -v /mnt/c/Users/Gebruiker/AppData/Local/Microsoft/Windows/Fonts:/usr/share/fonts/truetype/windows:ro \
  -w /workspace \
  loom-dev bash

# in the container run below command
apt update
apt install -y fontconfig

mkdir -p build
cd build
cmake ..
make -j"$(nproc)"
cd ..

SyntaxError: invalid syntax (2643177437.py, line 2)

In [ ]:
# after updating source
cd build && make -j$(nproc) transitmap && cd .. 

# to make a map
cat examples/netkaart3.json | ./build/topo | ./build/loom | ./build/octi -g 400 --geo-pen 0.1 --nd-move-pen 0.5 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > finaloutput.svg